# GPT-2 style model with grouped-query attention (GQA) + our own BPE tokenizer
Trains on Tiny Shakespeare and prints generated text.

**Before you run:** Runtime -> Change runtime type -> **T4 GPU**. Then run the cells top to bottom.

## 1. Get the code

In [ ]:
import os, subprocess

# Option A: clone your GitHub repo (edit the URL). Option B: upload gpt2-gqa.zip when prompted.
REPO_URL = "https://github.com/YOUR_USER/gpt2-gqa.git"

if not os.path.exists("/content/gpt2-gqa"):
    if "YOUR_USER" not in REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, "/content/gpt2-gqa"], check=True)
    else:
        from google.colab import files
        up = files.upload()  # choose gpt2-gqa.zip
        subprocess.run(["unzip", "-q", next(iter(up)), "-d", "/content"], check=True)

%cd /content/gpt2-gqa
!pip install -q -e . --no-deps

In [ ]:
import torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (switch the runtime to a GPU)")

## 2. Run the tests first
The model code has a test suite, including a check that grouped-query attention matches an independent per-head reference. **All tests should pass before you train.**

In [ ]:
!pytest -q

## 3. Data: download Tiny Shakespeare, train the BPE tokenizer, write token files

In [ ]:
!python -m gpt2_gqa.data --out_dir data/shakespeare --vocab_size 1024

## 4. Train with GQA (6 query heads sharing 2 key/value heads)
Takes a few minutes on a GPU (I could not time it, so treat that as an estimate). Validation loss is printed every 250 iterations.

In [ ]:
from gpt2_gqa.train import make_config, train

gqa = train(make_config("small", out_dir="out/gqa"))        # n_head=6, n_kv_head=2

## 5. Loss curve and generated text

In [ ]:
from IPython.display import Image, display
display(Image("out/gqa/loss_curve.png"))
print(open("out/gqa/samples.txt").read())

Generate more, with your own prompt:

In [ ]:
from gpt2_gqa.bpe import ByteBPE
from gpt2_gqa.model import load_checkpoint
from gpt2_gqa.generate import generate_text

device = "cuda" if torch.cuda.is_available() else "cpu"
model, ckpt = load_checkpoint("out/gqa/ckpt.pt", device)
tok = ByteBPE.load("out/gqa/tokenizer.json")
print(generate_text(model, tok, "ROMEO:", max_new_tokens=300, temperature=0.8, top_k=50, device=device, seed=0))

## 6. (Optional) Compare with standard multi-head attention
Same model, but every query head has its own key/value head (`n_kv_head=6`).

In [ ]:
mha = train(make_config("small", out_dir="out/mha", n_kv_head=6))

print(f"{'':14}{'params':>12}{'kv cache/token':>16}{'best val loss':>15}")
for name, s in (("GQA (6q/2kv)", gqa), ("MHA (6q/6kv)", mha)):
    print(f"{name:14}{s['params']:>12,}{s['kv_cache_bytes_per_token_fp16']:>14,} B{s['best_val_loss']:>15.4f}")

## 7. Download the results

In [ ]:
!zip -qr results.zip out
from google.colab import files
files.download("results.zip")